# Lesson 02 - Εξερεύνηση του Microsoft Agent Framework

Το **Microsoft Agent Framework (MAF)** είναι ένα ενοποιημένο πλαίσιο για τη δημιουργία πρακτόρων τεχνητής νοημοσύνης. Παρέχει μια καθαρή, συνθετική αρχιτεκτονική με τέσσερα βασικά δομικά στοιχεία:

- **Client** – συνδέεται με ένα τερματικό σημείο μοντέλου AI και διαχειρίζεται την επικοινωνία
- **Agent** – περιτυλίγει έναν client με οδηγίες και ορισμούς εργαλείων
- **Tools** – επεκτείνουν τις δυνατότητες του πράκτορα με προσαρμοσμένες λειτουργίες που μπορεί να καλεί το μοντέλο
- **Session** – διατηρεί το ιστορικό συνομιλιών για αλληλεπιδράσεις πολλαπλών γύρων

Σε αυτό το μάθημα, θα δημιουργήσουμε έναν **πράκτορα κρατήσεων ταξιδιών** που ελέγχει τη διαθεσιμότητα προορισμών χρησιμοποιώντας αυτές τις έννοιες.


## Ρύθμιση


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Κατανόηση της Αρχιτεκτονικής του Πλαισίου Agent

Το Microsoft Agent Framework ακολουθεί μια πολυεπίπεδη αρχιτεκτονική:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Πελάτης** – Ένας `AzureAIProjectAgentProvider` συνδέεται με μια ανάπτυξη Azure OpenAI. Διαχειρίζεται την αυθεντικοποίηση, τη μορφοποίηση των αιτημάτων και την ανάλυση των απαντήσεων.
2. **Agent** – Δημιουργείται από τον πελάτη μέσω του `provider.create_agent()`, ο agent συνδυάζει την πρόσβαση στο μοντέλο με οδηγίες (σύστημα prompt) και εργαλεία.
3. **Εργαλεία** – Συναρτήσεις Python διακοσμημένες με `@tool` που ο agent μπορεί να καλεί για να εκτελέσει ενέργειες ή να αντλήσει δεδομένα.
4. **Συνεδρία** – Ένα αντικείμενο `AgentSession` (δημιουργημένο μέσω `agent.create_session()`) που αποθηκεύει το ιστορικό συνομιλίας, επιτρέποντας διάλογο πολλών γύρων όπου ο agent θυμάται το προηγούμενο πλαίσιο.

Ας χτίσουμε κάθε επίπεδο βήμα προς βήμα.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Προσθήκη Εργαλείων με τον Διακοσμητή @tool

Τα εργαλεία επιτρέπουν στους πράκτορες να εκτελούν ενέργειες πέρα από την παραγωγή κειμένου. Ο διακοσμητής `@tool` μετατρέπει μια κανονική συνάρτηση Python σε κάτι που μπορεί να καλέσει ο πράκτορας.

Βασικά σημεία:
- Χρησιμοποιήστε `Annotated[type, "description"]` ώστε το μοντέλο να κατανοεί κάθε παράμετρο.
- Το docstring γίνεται η περιγραφή του εργαλείου που βλέπει το μοντέλο.
- Το `approval_mode="never_require"` σημαίνει ότι το εργαλείο εκτελείται αυτόματα χωρίς επιβεβαίωση από τον χρήστη.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Δημιουργία ενός Πράκτορα με Εργαλεία

Τώρα συνδυάζουμε τον πελάτη, τις οδηγίες και τα εργαλεία σε έναν πράκτορα. Οι `οδηγίες` λειτουργούν ως η προτροπή του συστήματος — ορίζουν την προσωπικότητα και τη συμπεριφορά του πράκτορα.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Συνομιλίες Πολλαπλών Γύρων με Συνεδρίες

Μια `AgentSession` (δημιουργημένη μέσω `agent.create_session()`) παρακολουθεί όλα τα μηνύματα σε μια συνομιλία. Με το να περνάμε την ίδια συνεδρία σε κάθε κλήση `agent.run()`, ο πράκτορας έχει πρόσβαση σε ολόκληρο το ιστορικό της συνομιλίας και μπορεί να ανατρέξει σε προηγούμενα μηνύματα.

Περνάμε `tools=[check_destination_availability]` ώστε ο πράκτορας να μπορεί να καλεί τον ελεγκτή διαθεσιμότητας κατά τη διάρκεια κάθε γύρου.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Περίληψη

Σε αυτό το μάθημα εξερευνήσατε τους τέσσερις πυλώνες του Microsoft Agent Framework:

| Έννοια | Τι Μάθατε |
|---------|------------------|
| **Πελάτης** | `AzureAIProjectAgentProvider` συνδέεται με το Azure OpenAI με αυθεντικοποίηση βασισμένη σε διαπιστευτήρια |
| **Πράκτορας** | `provider.create_agent()` συνδυάζει μια σύνδεση μοντέλου με οδηγίες και όνομα |
| **Εργαλεία** | Ο διακοσμητής `@tool` εκθέτει συναρτήσεις Python για να καλεί ο πράκτορας |
| **Συνεδρία** | `agent.create_session()` διατηρεί το ιστορικό συνομιλίας σε πολλούς γύρους |

Αυτά τα δομικά στοιχεία συνδυάζονται για να δημιουργήσουν πράκτορες που μπορούν να διεξάγουν φυσικές συνομιλίες, να καλούν εξωτερικές συναρτήσεις και να διατηρούν το πλαίσιο — το θεμέλιο για πιο προχωρημένα πρότυπα πρακτόρων σε επόμενα μαθήματα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Απόρρητο**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτόματες μεταφράσεις ενδέχεται να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στη γλώσσα του πρέπει να θεωρείται ως η αυθεντική πηγή. Για κρίσιμες πληροφορίες συνιστάται επαγγελματική μετάφραση από άνθρωπο. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
